## Usage Guide

**Best ways to interact with this chatbot:**

### 1. **Single Query Cell** (Recommended for focused questions)
- Scroll to the "Interactive Chat" section
- Edit the `query` variable with your question
- Run the cell to get a response with sources

### 2. **Multi-Query Testing** (Good for batch testing)
- Use the "Quick Multi-Query Testing" section
- Add multiple queries to the list
- Run once to get all responses

### 3. **Retrieval Analysis** (Debug/tune retrieval)
- Use the "Retrieval Analysis" section
- See exactly what documents are being retrieved
- Useful for understanding why you got certain responses

### 4. **CLI Mode** (For interactive conversation)
- Exit the notebook
- Run: `python gemini_rag_chatbot.py`
- This gives you a true multi-turn chat interface

**Configuration Tips:**
- Modify `RAG_TOP_K` in `.env` to retrieve more/fewer documents
- Adjust `RAG_SCORE_THRESHOLD` to make retrieval more/less strict
- Change `TEMPERATURE` to make responses more deterministic (0.0) or creative (2.0)

In [ ]:
query = "communication tips"

print(f"Query: '{query}'")
print(f"Top K: {rag.ChatbotConfig.TOP_K}")
print(f"Score Threshold: {rag.ChatbotConfig.SCORE_THRESHOLD}\n")

docs = retriever.get_relevant_documents(query)

print(f"Retrieved {len(docs)} documents:\n")
for i, doc in enumerate(docs, 1):
    print(f"{i}. {doc.page_content}")
    print("-" * 70)

## Retrieval Analysis

See what documents are being retrieved for a query:

In [1]:
queries = [
    "What are some good conversation starters?",
    "How can I improve communication with my partner?",
    "What are some romantic ideas?",
]

for i, q in enumerate(queries, 1):
    print(f"\n{'='*70}")
    print(f"Query {i}: {q}")
    print('='*70)
    
    try:
        response = chain.invoke(q)
        print(f"\nResponse:\n{response}")
    except Exception as e:
        print(f"Error: {e}")


Query 1: What are some good conversation starters?
Error: name 'chain' is not defined

Query 2: How can I improve communication with my partner?
Error: name 'chain' is not defined

Query 3: What are some romantic ideas?
Error: name 'chain' is not defined


## Quick Multi-Query Testing

Run multiple queries in one cell. Edit the queries list below:

# Gemini RAG Chatbot - Full Test Suite

This notebook tests all major components of the Gemini RAG chatbot without requiring interactive input.

In [2]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

import gemini_rag_chatbot as rag

print("✓ Imports successful")

/opt/homebrew/Caskroom/miniforge/base/envs/ai-text-opt/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Imports successful


## TEST 1: Environment Variable Loading

In [3]:
print("\n" + "=" * 70)
print("TEST 1: Environment Variable Loading")
print("=" * 70)

env_path = rag.ChatbotConfig.PROJECT_DIR / ".env"
print(f"Loading .env from: {env_path}")
print(f"File exists: {env_path.exists()}")

load_dotenv(dotenv_path=env_path)

api_key = os.getenv("GOOGLE_API_KEY")
print(f"✓ GOOGLE_API_KEY loaded: {'***' + api_key[-4:] if api_key else 'NOT FOUND'}")

gemini_model = os.getenv("GEMINI_MODEL")
print(f"✓ GEMINI_MODEL: {gemini_model}")

top_k = os.getenv("RAG_TOP_K")
print(f"✓ RAG_TOP_K: {top_k}")

if not api_key:
    print("❌ GOOGLE_API_KEY not found!")
    raise ValueError("Missing GOOGLE_API_KEY")

print("✅ Environment loading PASSED")


TEST 1: Environment Variable Loading
Loading .env from: /Users/adamaslan/code/ai-text-opt/.env
File exists: True
✓ GOOGLE_API_KEY loaded: ***Xcwk
✓ GEMINI_MODEL: gemini-2.0-flash-exp
✓ RAG_TOP_K: 5
✅ Environment loading PASSED


## TEST 2: Embeddings CSV Loading

In [4]:
print("\n" + "=" * 70)
print("TEST 2: Embeddings CSV Loading")
print("=" * 70)

csv_path = rag.ChatbotConfig.EMBEDDINGS_CSV
abs_path = rag.ChatbotConfig.PROJECT_DIR / csv_path
print(f"Loading from: {abs_path}")
print(f"File exists: {abs_path.exists()}")

try:
    texts, embeddings = rag.load_embeddings_from_csv(csv_path)
    print(f"✓ Loaded {len(texts)} documents")
    print(f"✓ Embeddings shape: {embeddings.shape}")
    print(f"✓ First text (truncated): {texts[0][:100]}...")
    print(f"✓ First embedding (first 5 dims): {embeddings[0][:5]}")
    print("✅ Embeddings loading PASSED")
except Exception as e:
    print(f"❌ Failed to load embeddings: {e}")
    raise


TEST 2: Embeddings CSV Loading
Loading from: /Users/adamaslan/code/ai-text-opt/embeddings/151qa2_with_embeddings.csv
File exists: True
✓ Loaded 367 documents
✓ Embeddings shape: (367, 384)
✓ First text (truncated): AI 151 doc qs data synthetic ideas...
✓ First embedding (first 5 dims): [-0.11337424  0.01502038 -0.00926142 -0.03565324 -0.0248416 ]
✅ Embeddings loading PASSED


## TEST 3: Vector Store Creation

In [5]:
print("\n" + "=" * 70)
print("TEST 3: Vector Store Creation")
print("=" * 70)

try:
    vector_store = rag.create_vector_store(texts, embeddings)
    print(f"✓ Vector store created")
    print(f"✓ FAISS index entries: {vector_store.index.ntotal}")
    print(f"✓ Index dimension: {vector_store.index.d}")
    print("✅ Vector store creation PASSED")
except Exception as e:
    print(f"❌ Failed to create vector store: {e}")
    raise


TEST 3: Vector Store Creation


/Users/adamaslan/code/ai-text-opt/gemini_rag_chatbot.py:160: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


✓ Vector store created
✓ FAISS index entries: 367
✓ Index dimension: 384
✅ Vector store creation PASSED


## TEST 4: LLM Initialization

In [6]:
print("\n" + "=" * 70)
print("TEST 4: LLM Initialization")
print("=" * 70)

try:
    llm = rag.create_gemini_llm()
    print(f"✓ LLM initialized: {rag.ChatbotConfig.GEMINI_MODEL}")
    print(f"✓ Temperature: {rag.ChatbotConfig.TEMPERATURE}")
    print(f"✓ Max tokens: {rag.ChatbotConfig.MAX_TOKENS}")
    print("✅ LLM initialization PASSED")
except ValueError as e:
    print(f"❌ Configuration error: {e}")
    raise
except Exception as e:
    print(f"❌ Failed to initialize LLM: {e}")
    raise


TEST 4: LLM Initialization
✓ LLM initialized: gemini-2.0-flash-exp
✓ Temperature: 0.7
✓ Max tokens: 512
✅ LLM initialization PASSED


## TEST 5: RAG Chain Assembly

In [7]:
print("\n" + "=" * 70)
print("TEST 5: RAG Chain Assembly")
print("=" * 70)

try:
    chain, retriever = rag.create_rag_chain(vector_store, llm)
    print(f"✓ RAG chain created")
    print(f"✓ Retriever configured with top_k={rag.ChatbotConfig.TOP_K}")
    print(f"✓ Score threshold: {rag.ChatbotConfig.SCORE_THRESHOLD}")
    print("✅ RAG chain assembly PASSED")
except Exception as e:
    print(f"❌ Failed to assemble RAG chain: {e}")
    raise


TEST 5: RAG Chain Assembly
✓ RAG chain created
✓ Retriever configured with top_k=5
✓ Score threshold: 0.3
✅ RAG chain assembly PASSED


## TEST 6: Retrieval Functionality

In [8]:
print("\n" + "=" * 70)
print("TEST 6: Retrieval Functionality")
print("=" * 70)

try:
    # Test with a simple query
    test_query = "question"
    docs = retriever.get_relevant_documents(test_query)
    print(f"✓ Retrieved {len(docs)} documents for query: '{test_query}'")
    if docs:
        print(f"✓ Top result (truncated): {docs[0].page_content[:100]}...")
    print("✅ Retrieval PASSED")
except Exception as e:
    print(f"❌ Retrieval failed: {e}")
    raise


TEST 6: Retrieval Functionality
❌ Retrieval failed: 'VectorStoreRetriever' object has no attribute 'get_relevant_documents'


AttributeError: 'VectorStoreRetriever' object has no attribute 'get_relevant_documents'

## TEST 7: Simple Query Test

In [9]:
print("\n" + "=" * 70)
print("TEST 7: Simple Query Test (Using RAG Chain)")
print("=" * 70)

test_query = "What is the most common question?"
print(f"\nQuery: {test_query}")
print("\nGenerating response...")

try:
    response = chain.invoke(test_query)
    print(f"\n✓ Response generated successfully")
    print(f"\nResponse:\n{response}")
    print("\n✅ Query test PASSED")
except Exception as e:
    print(f"❌ Query failed: {e}")
    raise


TEST 7: Simple Query Test (Using RAG Chain)

Query: What is the most common question?

Generating response...
❌ Query failed: Error calling model 'gemini-2.0-flash-exp' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-exp\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-exp\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash-exp\nPlease retry in 16.904391751s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.0-flash-exp' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-exp\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-exp\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash-exp\nPlease retry in 16.904391751s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini A

## TEST SUMMARY

In [ ]:
print("\n" + "=" * 70)
print("✅ ALL TESTS PASSED!")
print("=" * 70)
print("\n🎉 Chatbot is ready to use!")
print("\nTo run the interactive chatbot, use:")
print("  python gemini_rag_chatbot.py")
print("\n" + "=" * 70)

In [ ]:
# Edit this query and run the cell
query = "What are some creative ideas for intimacy?"

print(f"🤔 Query: {query}")
print("\n⏳ Retrieving context and generating response...\n")

# Get relevant documents
docs = retriever.get_relevant_documents(query)
print(f"📚 Retrieved {len(docs)} relevant documents\n")

# Generate response
response = chain.invoke(query)

print("🤖 Response:")
print("-" * 70)
print(response)
print("-" * 70)

# Show sources
if docs:
    print("\n📖 Sources Used:")
    for i, doc in enumerate(docs[:3], 1):
        excerpt = doc.page_content[:150].replace("\n", " ")
        print(f"\n{i}. {excerpt}...")

## Interactive Chat - Ask Your Questions

You can now query the chatbot! Modify the query below and run the cell to get a response.